# PaliGemma-3B: T circuit and J_PT (D77)

Fixed in **DECISIONS.md D77** before this notebook existed: Qwen's T design
(D71) and overlap rule (D70, D72), on PaliGemma, with D76/D76a's method (full
forward passes, several heads per pass, last-position logits).

1. **Objects:** the 302 T-conflicting objects already selected in
   `results/pg_selection_paligemma2_3b.json` (no re-selection).
2. **Order:** the ones also among PaliGemma's 121 P objects first, then the
   rest in seeded random order (`t_patching.sweep_order`, seed 0).
3. **Patching:** source = incongruent note, target = neutral note, every head,
   both directions.
4. **Saves** `pg_t_patching_paligemma2_3b.json` and
   `pg_t_patching_records_paligemma2_3b.jsonl.gz`. Download both into
   `results/` and run `scripts/analyze_pg_t_circuit.py` for the D70 verdict
   (it needs at least 60 objects, D77 (3)).

**Runtime: ~20 h for 302 objects, so several sessions.** Each session patches
for `PATCH_BUDGET_H` hours (section 5) and then saves. To continue, attach the
previous version's output (or `ckpt/records.jsonl` as a dataset) as an input
and run again: section 3 prints how many objects are already done. The save
cell runs after every session, so each version's output is a complete,
analysable snapshot.

**Untested on the GPU** beyond what `pg_replication.ipynb` exercised. Settings:
GPU T4, Internet on, `HF_TOKEN` ticked (with the PaliGemma 2 licence accepted).
Run as **Save Version -> Save & Run All**.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/SohamRShinde/VLM-Overlap-RP"
REPO_DIR = "/kaggle/working/research_paper"

import importlib, os, subprocess, sys

if REPO_URL:
    if os.path.exists(REPO_DIR):
        # Always update an existing checkout: a stale clone silently runs old
        # code, and the failure shows up later as a confusing ImportError.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "-q", "origin/main"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)

if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

# Drop any already-imported package modules: Python caches them, so after an
# update the kernel would keep serving the old file.
for _m in [m for m in list(sys.modules) if m.startswith("circuit_overlap")]:
    del sys.modules[_m]
importlib.invalidate_caches()

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("repo at", COMMIT)   # record this in RESULTS.md with any number from this run

# Fail here, loudly, if the checkout is older than this notebook expects.
import importlib.util as _ilu
_need = {}
for _mod, _names in {
    "circuit_overlap.paligemma": ["CHECKPOINT", "load_paligemma", "build_inputs", "text_geometry",
                                  "FullForwardRunner", "square_image_block",
                                  "last_token_reaches_earlier_positions", "patch_example_batched",
                                  "quiet_processor_warning"],
    "circuit_overlap.patching": ["patch_example", "mean_restoration_grids", "classify_heads",
                                 "capture_last_token_o_proj_inputs"],
    "circuit_overlap.cross_ablation": ["ablate_heads_last_token"],
    "circuit_overlap.replication": ["lietzow_rows", "run_cached"],
    "circuit_overlap.t_patching": ["render_t_pair", "is_t_conflicting", "sweep_order",
                                   "has_nonfinite", "finite_records", "LAYOUT", "T_ROLES"],
    "circuit_overlap.t_screen": ["PRIOR_PROMPT", "VISUAL_PROMPT", "color_first_token_ids",
                                 "load_notes_config", "native_bf16"],
    "circuit_overlap.heads": ["load_head_set", "format_head_label"],
    "circuit_overlap.metrics": ["jaccard", "containment"],
    "circuit_overlap.provenance": ["result_envelope", "write_result"],
}.items():
    if _ilu.find_spec(_mod) is None:
        _need[_mod] = ["module"]
        continue
    _m = importlib.import_module(_mod)
    _gone = [n for n in _names if not hasattr(_m, n)]
    if _gone:
        _need[_mod] = _gone
for _f in ("configs/heads_lietzow_paligemma2_3b.json", "results/pg_selection_paligemma2_3b.json",
           "results/pg_replication_patching_paligemma2_3b.json",
           "results/split_half_ceiling_p_paligemma2_3b.json", "scripts/analyze_pg_t_circuit.py"):
    if not os.path.exists(f"{REPO_DIR}/{_f}"):
        _need[_f] = ["file"]
import inspect as _insp
if not _need:   # D76a additions the name check cannot see
    from circuit_overlap.replication import run_cached as _rc
    from circuit_overlap.paligemma import FullForwardRunner as _ffr
    for _what, _ok in (("run_cached(budget_s=...)", "budget_s" in _insp.signature(_rc).parameters),
                       ("FullForwardRunner(last_only=...)", "last_only" in _insp.signature(_ffr).parameters)):
        if not _ok:
            _need[_what] = ["parameter"]
if _need:
    raise RuntimeError(f"checkout {COMMIT} is missing {_need}. "
                       "Push the local commits to origin/main, then re-run this cell.")
print("code is current for this notebook")

In [ ]:
# Use one GPU. On "GPU T4 x2", device_map="auto" splits the model across both
# cards, which only adds transfer time for a 3B model. This must run before
# torch is first imported, so restart the kernel if torch is already loaded.
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# bitsandbytes is required for the 4-bit load (D10). Installing it into a
# kernel that already imported transformers does nothing: restart and re-run.
!pip install -q -U "transformers>=4.49" accelerate datasets inflect "bitsandbytes>=0.46.1"
import torch, transformers
from transformers.utils import is_bitsandbytes_available
from circuit_overlap.t_screen import native_bf16
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))
# Not torch.cuda.is_bf16_supported(): it says True on a T4 (emulation).
print("bitsandbytes available:", is_bitsandbytes_available(),
      "| native bf16:", native_bf16(torch.cuda.get_device_capability(0)))

from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    # Secrets are attached PER NOTEBOOK: one on your account is not enough.
    raise RuntimeError(
        "HF_TOKEN is not attached to this notebook. In the editor: Add-ons -> "
        "Secrets -> tick the checkbox next to HF_TOKEN (exact label), then "
        "restart the session and run from cell 1. It must be ticked before "
        "Save & Run All, or the background run fails here."
    ) from exc

## 2. Model, geometry, and Lietzow's heads

As `pg_replication.ipynb`. Their heads are only used for the hook check in
section 3; the P set this notebook compares against is the re-discovered one
(identical to theirs, D76).

In [ ]:
import json
from circuit_overlap.paligemma import CHECKPOINT, load_paligemma, quiet_processor_warning, text_geometry
from circuit_overlap.heads import load_head_set, format_head_label
from circuit_overlap.t_screen import PRIOR_PROMPT, VISUAL_PROMPT

quiet_processor_warning()   # the per-call `<image>` nag; section 3 checks the input itself

model, processor = load_paligemma()
COMPUTE_DTYPE = str(model.config.quantization_config.bnb_4bit_compute_dtype).replace("torch.", "")
print(CHECKPOINT, "| compute dtype:", COMPUTE_DTYPE, "(D76 (3): fp16 on a T4)")
print("devices used:", sorted({str(d) for d in getattr(model, "hf_device_map", {"": model.device}).values()}))

geo = text_geometry(model)          # raises if o_proj's width != n_heads x head_dim
n_layers, n_heads, head_dim = geo["n_layers"], geo["n_heads"], geo["head_dim"]
print(geo)

HEADS_FILE = f"{REPO_DIR}/configs/heads_lietzow_paligemma2_3b.json"
hs = load_head_set(HEADS_FILE, n_layers=n_layers, n_heads_per_layer=n_heads)   # raises if out of range
hs_raw = json.load(open(HEADS_FILE))
print("their promoting :", [format_head_label(h) for h in hs.promoting])
print("their suppressing:", [format_head_label(h) for h in hs.suppressing])
print("they report:", hs_raw["group_ablation_flip_pct"], "on", hs_raw["n_correctly_conflicting"],
      "objects; single heads", hs.single_head_prior_flip_pct)

## 3. [VERIFY] the same checks as `pg_replication.ipynb`

Prompt and image block, `labels` dropped without changing the logits, hooks in
a full forward pass, the T renders, last-position-only logits, and the batch
size timing. Stops the notebook if any fails.

In [ ]:
import glob, inflect, pathlib, shutil, time
import torch
from collections import Counter
from IPython.display import display
from datasets import load_dataset
from circuit_overlap.paligemma import (
    FullForwardRunner, build_inputs as _pg_inputs, square_image_block,
    last_token_reaches_earlier_positions,
)
from circuit_overlap.patching import capture_last_token_o_proj_inputs
from circuit_overlap.cross_ablation import ablate_heads_last_token
from circuit_overlap.replication import lietzow_rows
from circuit_overlap.t_screen import color_first_token_ids, load_notes_config
from circuit_overlap.t_patching import LAYOUT, render_t_pair

WORK = pathlib.Path("/kaggle/working/pg_t_patching")
CKPT = WORK / "ckpt"
CKPT.mkdir(parents=True, exist_ok=True)
# Resume: bring back records.jsonl from an earlier run attached as an input
# (its notebook output, or the file uploaded as a dataset). Found automatically
# anywhere under /kaggle/input; set RESUME_FROM to a folder only if the search
# picks the wrong file.
RESUME_FROM = None          # e.g. "/kaggle/input/pg-t-patching-ckpt"
for name in ("records.jsonl",):
    if RESUME_FROM:
        found = [f"{RESUME_FROM}/{name}"] if os.path.exists(f"{RESUME_FROM}/{name}") else []
    else:
        found = (sorted(glob.glob(f"/kaggle/input/**/pg_t_patching/ckpt/{name}", recursive=True))
                 or sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True)))
    if found and not (CKPT / name).exists():
        shutil.copy(found[0], CKPT / name)
    n_lines = sum(1 for l in open(CKPT / name) if l.strip()) if (CKPT / name).exists() else 0
    print(f"{name}: {n_lines} objects already done" + (f" (from {found[0]})" if found else " (nothing to resume)"))

ds = load_dataset("mgolov/Visual-Counterfact", split="color")
_eng = inflect.engine()
rows, dropped = lietzow_rows(ds, _eng)
print(len(rows), "objects", dict(dropped), "(the Qwen runs used the same 469)")
vocab = sorted({c for r in rows for c in r["correct"]} | {r["attack"] for r in rows})
color_ids, collisions = color_first_token_ids(processor.tokenizer, vocab)
ambiguous = set(collisions)
print(len(vocab), "colours | sharing a first token on the Gemma tokenizer (D26):", collisions)

def build_inputs(image, prompt):
    return _pg_inputs(processor, image, prompt, model.device)

def prompts(r):
    return (VISUAL_PROMPT.format(object=r["name"]),
            PRIOR_PROMPT.format(a_object=_eng.a(r["name"])))

notes_cfg = load_notes_config(f"{REPO_DIR}/configs/notes.json")
spec = next(s for s in notes_cfg["specs"] if s.name == LAYOUT)
neutral_words = notes_cfg["neutral_words"]

def t_inputs(r):
    """(incongruent inputs, neutral inputs, render info), as t_patching.ipynb."""
    row = ds[r["i"]]
    pr = render_t_pair(row["original_image"], row["counterfact_image"], r["attack"], neutral_words, spec)
    prompt = VISUAL_PROMPT.format(object=r["name"])
    return build_inputs(pr["incongruent"], prompt), build_inputs(pr["neutral"], prompt), pr

VERIFY = {"compute_dtype": COMPUTE_DTYPE, "geometry": geo, "ambiguous_colours": sorted(ambiguous)}
IMAGE_TOKEN_ID = processor.image_token_id      # their get_image_token_id

# [VERIFY] 1-2. the prompt and the image block.
r0 = rows[0]
inp0 = build_inputs(ds[r0["i"]]["counterfact_image"], prompts(r0)[1])
ids0 = inp0["input_ids"][0].tolist()
offset0, side0 = square_image_block(ids0, IMAGE_TOKEN_ID)
text0 = processor.tokenizer.decode(ids0[offset0 + side0 * side0:])
VERIFY["image_block"] = {"offset": offset0, "side": side0, "n_tokens": len(ids0),
                         "pixel_values": list(inp0["pixel_values"].shape)}
VERIFY["prompt_after_image"] = text0
print("inputs:", {k: tuple(v.shape) for k, v in inp0.items()})
print("image block:", VERIFY["image_block"])
print("text after the image:", repr(text0))

# [VERIFY] 7. labels: the first run passed them; this run drops them.
inp0_labels = _pg_inputs(processor, ds[r0["i"]]["counterfact_image"], prompts(r0)[1], model.device,
                         keep_labels=True)
VERIFY["processor_returns_labels"] = "labels" in inp0_labels
if VERIFY["processor_returns_labels"]:
    with torch.no_grad():
        with_labels = model(**inp0_labels, use_cache=False).logits[0, -1].float()
        without = model(**inp0, use_cache=False).logits[0, -1].float()
    VERIFY["labels_max_dlogit"] = float((with_labels - without).abs().max())
    print("logits with vs without labels, max |dlogit|:", VERIFY["labels_max_dlogit"], "(expect ~0)")
    assert VERIFY["labels_max_dlogit"] < 0.05, (
        "labels change the logits (attention mask): the first run's checkpoints are not comparable -- "
        "stop and report this")
    del with_labels, without
del inp0_labels
assert ids0[-1] != IMAGE_TOKEN_ID, "the last token must be text"
assert prompts(r0)[1] in text0, "the prompt is not what the model reads"

# [VERIFY] 3. colour tokens: colliding colours are excluded (D26), and reported.
print("excluded colours (D26):", sorted(ambiguous) or "none")

# [VERIFY] 4. hooks in a full forward pass.
runner0 = FullForwardRunner(model, inp0)
base0 = runner0.logits()
with ablate_heads_last_token(model, [], head_dim):
    same0 = runner0.logits()
with ablate_heads_last_token(model, hs.promoting, head_dim):
    knocked0 = runner0.logits()
with capture_last_token_o_proj_inputs(model) as acts0:
    runner0.logits()
VERIFY["empty_ablation_max_dlogit"] = float((same0 - base0).abs().max())
VERIFY["promoting_ablation_max_dlogit"] = float((knocked0 - base0).abs().max())
print("empty ablation max |dlogit|:", VERIFY["empty_ablation_max_dlogit"], "(expect ~0)")
print("promoting ablation max |dlogit|:", VERIFY["promoting_ablation_max_dlogit"], "(expect > 0)")
assert VERIFY["empty_ablation_max_dlogit"] < 1e-3
assert VERIFY["promoting_ablation_max_dlogit"] > 1e-3
assert sorted(acts0) == list(range(n_layers)) and all(
    acts0[l].shape[-1] == n_heads * head_dim for l in acts0), "captured head outputs have the wrong shape"
del acts0

# [VERIFY] 5. the T renders.
s0_, n0_, pr0 = t_inputs(r0)
print(f"{r0['name']}: note {r0['attack']!r} vs neutral {pr0['neutral_word']!r}")
display(pr0["neutral"].resize((256, 256))); display(pr0["incongruent"].resize((256, 256)))
assert pr0["same_note_footprint"], "the two notes differ in shape: not a single-factor contrast"
assert s0_["input_ids"].shape == n0_["input_ids"].shape, "renders tokenise to different lengths"

# Attention direction (reported, D76 (2)).
VERIFY["last_token_vs_position_0"] = last_token_reaches_earlier_positions(model, inp0, position=0)
VERIFY["last_token_vs_first_text"] = last_token_reaches_earlier_positions(
    model, inp0, position=offset0 + side0 * side0)
for k in ("last_token_vs_position_0", "last_token_vs_first_text"):
    print(k, VERIFY[k])

# [VERIFY] 6. last-position-only logits (runner default) == the full output's last row.
full0 = FullForwardRunner(model, inp0, last_only=False).logits()
VERIFY["logits_to_keep_max_dlogit"] = float((full0 - base0).abs().max())
print("last-only vs full logits, max |dlogit|:", VERIFY["logits_to_keep_max_dlogit"], "(expect ~0)")
assert VERIFY["logits_to_keep_max_dlogit"] < 0.05

# Speed, and the total, before any long loop (CLAUDE.md).
from circuit_overlap.paligemma import _repeat_rows

def _time(fn, reps=2):
    fn(); torch.cuda.synchronize(); t0 = time.time()
    for _ in range(reps):
        fn()
    torch.cuda.synchronize()
    return (time.time() - t0) / reps

step = _time(runner0.logits)
print(f"one forward pass (last-only logits): {step:.2f}s")
VERIFY["forward_s"] = step
per_row = {1: step}
for b in (4, 8, 16):
    try:
        batch_in = _repeat_rows(inp0, b)
        with torch.no_grad():
            per_row[b] = _time(lambda: model(**batch_in, use_cache=False, logits_to_keep=1)) / b
        print(f"batch {b:2d}: {per_row[b]:.2f}s per row")
    except torch.cuda.OutOfMemoryError:
        print(f"batch {b:2d}: out of memory")
        break
    finally:
        batch_in = None
        torch.cuda.empty_cache()
BATCH = min(per_row, key=per_row.get)
VERIFY["per_row_s"], VERIFY["batch_size"] = per_row, BATCH
n_t = 302                                          # D77 (2), for this estimate only
print(f"batch size {BATCH}. estimate: ~{(2 * step + 2 * n_layers * n_heads * per_row[BATCH]) / 60:.1f} min "
      f"per object, ~{n_t * (2 * step + 2 * n_layers * n_heads * per_row[BATCH]) / 3600:.1f} h for {n_t}")
del runner0, base0, same0, knocked0, full0

## 4. The 302 objects and their order (D77 (2), (3))

Read from the committed selection file, recomputed with this run's colour
tokens; the count must be D77's 302. P's objects come from the committed
PaliGemma patching file.

In [ ]:
import json
from circuit_overlap.t_patching import is_t_conflicting, sweep_order, T_ROLES

SEL = json.load(open(f"{REPO_DIR}/results/pg_selection_paligemma2_3b.json"))["result"]
assert {k: sorted(v) for k, v in SEL["color_ids"].items()} == {k: sorted(v) for k, v in color_ids.items()}, \
    "colour tokens differ from the selection run: the tokenizer changed"
row_by_i = {r["i"]: r for r in rows}
selected_top1 = {s["i"]: (s["top1_incongruent"], s["top1_neutral"]) for s in SEL["records"]
                 if s["status"] == "ok"
                 and is_t_conflicting(row_by_i[s["i"]], s["top1_neutral"], s["top1_incongruent"],
                                      color_ids, ambiguous)}
selected = sorted(selected_top1)
assert len(selected) == 302, f"{len(selected)} T objects, D77 says 302"

P_FILE = f"{REPO_DIR}/results/pg_replication_patching_paligemma2_3b.json"
p_ids = [r["i"] for r in json.load(open(P_FILE))["result"]["records"]]
order = sweep_order(selected, p_ids, seed=0)
todo = [row_by_i[i] for i in order]
n_shared = len(set(selected) & set(p_ids))
print(f"{len(todo)} T objects; the first {n_shared} are also P objects | roles: {T_ROLES}")

## 5. Patching sweep (D77 (4))

First a check that batched patching reproduces `patch_example` on one full
layer of the first T object (D76a). Then every remaining object, both
directions over every head, until `PATCH_BUDGET_H` hours have passed. Each
object's top-1 pair is re-checked against the selection file first
(`selection_drift` if it moved).

In [ ]:
from circuit_overlap.patching import patch_example
from circuit_overlap.paligemma import patch_example_batched
from circuit_overlap.replication import run_cached
from circuit_overlap.t_patching import has_nonfinite

PATCH_BUDGET_H = 11.0     # hours of patching this session (Kaggle stops a session at 12 h)

# [VERIFY] batched == unbatched on one full layer of the first T object.
src1, tgt1, _ = t_inputs(todo[0])
check_layer = hs.promoting[0][0]          # a layer that matters, taken from their list
slow = patch_example(model, FullForwardRunner(model, src1), FullForwardRunner(model, tgt1), head_dim,
                     layers=[check_layer])
fast = patch_example_batched(model, src1, tgt1, head_dim, batch_size=BATCH, layers=[check_layer])
diff = max(abs(a - b) for k in ("p2v_logit_prior", "p2v_logit_visual", "v2p_logit_prior", "v2p_logit_visual")
           for a, b in zip(slow[k][check_layer], fast[k][check_layer]))
VERIFY["batched_vs_unbatched_max_dlogit"] = diff
print(f"batched vs unbatched, layer {check_layer}: max |dlogit| {diff:.4f}")
assert slow["next_token_idx_prior"] == fast["next_token_idx_prior"]
assert diff < 0.5, "batched patching does not reproduce patch_example: do not run the sweep"
del slow, fast, src1, tgt1

def sweep_object(r):
    src_in, tgt_in, _ = t_inputs(r)
    if src_in["input_ids"].shape != tgt_in["input_ids"].shape:
        return {"status": "shape_mismatch", "name": r["name"]}
    top1 = (int(FullForwardRunner(model, src_in).logits().argmax()),
            int(FullForwardRunner(model, tgt_in).logits().argmax()))
    out = {"name": r["name"], "top1_incongruent": top1[0], "top1_neutral": top1[1]}
    if top1 != selected_top1[r["i"]]:
        return {"status": "selection_drift", **out}
    # Roles (T_ROLES): source = incongruent note, target = neutral note.
    record = patch_example_batched(model, src_in, tgt_in, head_dim, batch_size=BATCH)
    if record is None:                                     # argmax tokens coincide
        return {"status": "same_argmax", **out}
    for key, value in record.items():
        if isinstance(value, list):
            record[key] = [[round(x, 4) for x in row] for row in value]
    full = {"status": "patched", **out, **record}
    if has_nonfinite(full):
        full["status"] = "numerics_failed"                 # plan risk 7
    return full

sweep = run_cached(CKPT / "records.jsonl", todo, sweep_object, progress_every=5,
                   budget_s=PATCH_BUDGET_H * 3600)
print(Counter(s["status"] for s in sweep), f"-- {len(sweep)} of {len(todo)} T objects done")

## 6. Classify and save (after every session)

Saves a complete snapshot of everything done so far, so each version's output
can be analysed or resumed. The overlap verdict is the committed script's
(`scripts/analyze_pg_t_circuit.py`); the printout here is a **preliminary**
run of it, and only once at least 60 objects are patched (D77 (3)).

In [ ]:
import gzip, sys
from circuit_overlap.patching import mean_restoration_grids, classify_heads
from circuit_overlap.t_patching import finite_records
from circuit_overlap.metrics import jaccard, containment
from circuit_overlap.provenance import result_envelope, write_result

patched, nonfinite = finite_records([s for s in sweep if s["status"] == "patched"])
summary = {"n_selected": len(selected), "n_done": len(sweep), "n_scored": len(patched),
           "sweep_status": dict(Counter(s["status"] for s in sweep)), "nonfinite_ids": nonfinite,
           "order_seed": 0, "n_shared_with_p": n_shared, "layout": LAYOUT, "roles": T_ROLES,
           "verify": VERIFY, "records_file": "pg_t_patching_records_paligemma2_3b.jsonl.gz"}
if patched:
    p2v, v2p, n_used = mean_restoration_grids(patched)
    cls = classify_heads(p2v, v2p, sigma=2.0)
    t_prom = set(cls["promoting"])
    p_prom = {tuple(h) for h in json.load(open(P_FILE))["result"]["promoting"]}
    print(f"{n_used} objects scored | T promoting:", sorted(format_head_label(h) for h in t_prom))
    print("P promoting:", sorted(format_head_label(h) for h in p_prom))
    print(f"PRELIMINARY J_PT {jaccard(p_prom, t_prom):.3f} | containment P in T {containment(p_prom, t_prom):.2f}")
    summary.update({
        "p2v_grid": p2v.round(6).tolist(), "v2p_grid": v2p.round(6).tolist(),
        "pc1_scores": cls["pc1_scores"].round(6).tolist(), "pc1_direction": cls["pc1_direction"],
        "pc1_mean": cls["pc1_mean"], "pc1_std": cls["pc1_std"], "sigma": cls["sigma"],
        "promoting": [list(h) for h in sorted(t_prom)],
        "suppressing": [list(h) for h in sorted(cls["suppressing"])]})

print(write_result(WORK / "pg_t_patching_paligemma2_3b.json", result_envelope(
    "pg_t_patching", summary, model="PaliGemma2-3B-mix-448", checkpoint=CHECKPOINT, seed=0,
    prompt=f"visual={VISUAL_PROMPT!r}; incongruent vs neutral {LAYOUT} note", n=len(patched),
    repo_dir=REPO_DIR, decision="D77", quantisation="4-bit nf4, double quant",
    compute_dtype=COMPUTE_DTYPE, gpu=torch.cuda.get_device_name(0), image_tokens=side0 * side0,
    method="full forward pass per patched head, batched (D76a); Lietzow PCA +/-2 sigma; "
           "source=incongruent note, target=neutral note")))
with gzip.open(WORK / "pg_t_patching_records_paligemma2_3b.jsonl.gz", "wt") as f:
    for s in sweep:
        f.write(json.dumps(s) + "\n")
for name in ("pg_t_patching_paligemma2_3b.json", "pg_t_patching_records_paligemma2_3b.jsonl.gz"):
    print(f"  {name:46s} {(WORK / name).stat().st_size / 1e6:6.2f} MB")

if len(patched) >= 60:
    sys.path.insert(0, f"{REPO_DIR}/scripts")
    from analyze_pg_t_circuit import main as d77_overlap
    print("\nPRELIMINARY (the committed script on this session's files):\n")
    d77_overlap(P_FILE, str(WORK / "pg_t_patching_records_paligemma2_3b.jsonl.gz"),
                f"{REPO_DIR}/results/split_half_ceiling_p_paligemma2_3b.json")
else:
    print(f"\n{len(patched)} objects patched: fewer than 60 (D77 (3)), no preliminary verdict yet")
print(f"\n{len(sweep)} of {len(todo)} done. To continue: attach this version's output and run again.")